In [ ]:
# ==============================================================================
# TASK-CONDITIONED UNIFIED MODEL
# ==============================================================================

import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# ------------------------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------------------------

MODEL_NAME = "dicta-il/dictabert"
TRAIN_FILE = "train_set.csv"
VAL_FILE = "val_set.csv"
OUTPUT_DIR = "/content/models/task_conditioned_kreitler5"
MAX_LEN = 160

ATTRIBUTE_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

O = 0
B_MU = 1
I_MU = 2

# ------------------------------------------------------------------------------
# 2. ENCODERS & INVERSE CLASS WEIGHTS
# ------------------------------------------------------------------------------

train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)

encoders = {}
decoders = {}
num_classes = {}

for col in ATTRIBUTE_COLS:
    values = train_df[col].dropna().astype(str).unique().tolist()
    encoders[col] = {value: i for i, value in enumerate(values)}
    decoders[col] = {i: value for value, i in encoders[col].items()}
    num_classes[col] = len(encoders[col])


def compute_class_weights(df, column, encoder):
    valid_values = df[column].dropna().astype(str)
    counts = valid_values.map(encoder).value_counts()
    n_classes = len(encoder)
    total = len(valid_values)

    weights = torch.ones(n_classes, dtype=torch.float)
    for class_id in range(n_classes):
        count = counts.get(class_id, 0)
        if count > 0:
            weights[class_id] = total / (n_classes * count)

    return torch.clamp(weights, min=0.1, max=10.0)


class_weights = {
    col: compute_class_weights(train_df, col, encoders[col])
    for col in ATTRIBUTE_COLS
}

# ------------------------------------------------------------------------------
# 3. TOKENIZER SETUP & SPECIAL PROMPT TOKENS
# ------------------------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
SPECIAL_TOKENS = {
    "additional_special_tokens": [
        "<mv>",
        "</mv>",
        "[TASK:MV]",
        "[TASK:REF]",
        "[TASK:ATTR]",
    ]
}
tokenizer.add_special_tokens(SPECIAL_TOKENS)

# ------------------------------------------------------------------------------
# 4. DATASET WITH STRICT REFERENT ALIGNMENT
# ------------------------------------------------------------------------------


class TaskConditionedDataset(Dataset):

    def __init__(self, df, tokenizer, encoders, max_len=160):
        self.tokenizer = tokenizer
        self.encoders = encoders
        self.max_len = max_len
        self.examples = []

        grouped = df.groupby(["Prompt_Word", "Full_Raw_Sentence"], sort=False)

        for (prompt_word, sentence), group in grouped:
            if pd.isna(sentence):
                continue
            sentence = str(sentence).strip()
            prompt_word = (
                str(prompt_word).strip() if pd.notna(prompt_word) else ""
            )

            # Collect all Meaning Value character spans for Task 1
            all_mv_spans = []
            for _, row in group.iterrows():
                if pd.notna(row["MeaningVal"]):
                    mv_str = str(row["MeaningVal"]).strip()
                    pos = sentence.find(mv_str)
                    if pos != -1:
                        all_mv_spans.append((pos, pos + len(mv_str)))

            all_mv_spans = list(dict.fromkeys(all_mv_spans))

            if all_mv_spans:
                self.examples.append({
                    "task_type": "mv",
                    "sentence": sentence,
                    "prompt_word": prompt_word,
                    "all_mv_spans": all_mv_spans,
                })

            for _, row in group.iterrows():
                if pd.isna(row["MeaningVal"]):
                    continue
                mv = str(row["MeaningVal"]).strip()
                ref = (
                    str(row["Referent"]).strip()
                    if pd.notna(row["Referent"])
                    else ""
                )

                self.examples.append({
                    "task_type": "ref",
                    "sentence": sentence,
                    "prompt_word": prompt_word,
                    "meaning_val": mv,
                    "referent": ref,
                })

                self.examples.append({
                    "task_type": "attr",
                    "sentence": sentence,
                    "meaning_val": mv,
                    "referent": ref,
                    "attributes": {col: row[col] for col in ATTRIBUTE_COLS},
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        task_type = ex["task_type"]

        if task_type == "mv":
            text = f"[TASK:MV] משפט: {ex['sentence']}"
            encoding = self.tokenizer(
                text,
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_offsets_mapping=True,
                return_tensors="pt",
            )
            input_ids = encoding["input_ids"].squeeze(0)
            attention_mask = encoding["attention_mask"].squeeze(0)
            offsets = encoding["offset_mapping"].squeeze(0).tolist()

            prefix_len = len("[TASK:MV] משפט: ")
            sentence_st = prefix_len
            sentence_en = prefix_len + len(ex["sentence"])

            bio_labels = torch.full((self.max_len,), -100, dtype=torch.long)
            for t_idx, (st, en) in enumerate(offsets):
                if st == 0 and en == 0:
                    continue
                if en <= sentence_st or st >= sentence_en:
                    continue
                bio_labels[t_idx] = O

            for mv_st, mv_en in ex["all_mv_spans"]:
                mv_st += sentence_st
                mv_en += sentence_st
                first = True
                for t_idx, (st, en) in enumerate(offsets):
                    if st == 0 and en == 0:
                        continue
                    if en > mv_st and st < mv_en:
                        bio_labels[t_idx] = B_MU if first else I_MU
                        first = False

            return {
                "task_type": 1,
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "bio_labels": bio_labels,
                "ref_start": torch.tensor(-100, dtype=torch.long),
                "ref_end": torch.tensor(-100, dtype=torch.long),
                "attr_labels": {
                    col: torch.tensor(-100, dtype=torch.long)
                    for col in ATTRIBUTE_COLS
                },
            }

        elif task_type == "ref":
            mv = ex["meaning_val"]
            sent = ex["sentence"]
            marked_sent = (
                sent.replace(mv, f"<mv> {mv} </mv>", 1) if mv in sent else sent
            )

            context = f"[TASK:REF] גירוי: {ex['prompt_word']} | משפט: {marked_sent}"
            question = f"מהו הרפרנט של יחידת המשמעות: {mv} ?"

            encoding = self.tokenizer(
                question,
                context,
                max_length=self.max_len,
                padding="max_length",
                truncation="only_second",
                return_offsets_mapping=True,
                return_tensors="pt",
            )
            input_ids = encoding["input_ids"].squeeze(0)
            attention_mask = encoding["attention_mask"].squeeze(0)

            ref = ex["referent"]
            ref_st, ref_en = -100, -100

            # Match exact referent string inside context.
            if ref and ref.strip():
                ref_c_st = context.find(ref)
                if ref_c_st != -1:
                    ref_c_en = ref_c_st + len(ref)
                    st_tok = encoding.char_to_token(
                        0, ref_c_st, sequence_index=1
                    )
                    en_tok = encoding.char_to_token(
                        0, ref_c_en - 1, sequence_index=1
                    )
                    ref_st = st_tok if st_tok is not None else -100
                    ref_en = en_tok if en_tok is not None else -100

            return {
                "task_type": 2,
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "bio_labels": torch.full((self.max_len,), -100, dtype=torch.long),
                "ref_start": torch.tensor(ref_st, dtype=torch.long),
                "ref_end": torch.tensor(ref_en, dtype=torch.long),
                "attr_labels": {
                    col: torch.tensor(-100, dtype=torch.long)
                    for col in ATTRIBUTE_COLS
                },
            }

        else:
            text = f"[TASK:ATTR] משפט: {ex['sentence']} | יחידה: {ex['meaning_val']} | רפרנט: {ex['referent']}"
            encoding = self.tokenizer(
                text,
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            input_ids = encoding["input_ids"].squeeze(0)
            attention_mask = encoding["attention_mask"].squeeze(0)

            attr_labels = {}
            for col in ATTRIBUTE_COLS:
                val = ex["attributes"][col]
                if pd.isna(val):
                    attr_labels[col] = torch.tensor(-100, dtype=torch.long)
                else:
                    cid = self.encoders[col].get(str(val), -100)
                    attr_labels[col] = torch.tensor(cid, dtype=torch.long)

            return {
                "task_type": 3,
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "bio_labels": torch.full((self.max_len,), -100, dtype=torch.long),
                "ref_start": torch.tensor(-100, dtype=torch.long),
                "ref_end": torch.tensor(-100, dtype=torch.long),
                "attr_labels": attr_labels,
            }


# ------------------------------------------------------------------------------
# 5. DATA COLLATOR
# ------------------------------------------------------------------------------


def task_conditioned_collate_fn(batch):
    return {
        "task_types": torch.tensor(
            [x["task_type"] for x in batch], dtype=torch.long
        ),
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "bio_labels": torch.stack([x["bio_labels"] for x in batch]),
        "ref_start": torch.stack([x["ref_start"] for x in batch]),
        "ref_end": torch.stack([x["ref_end"] for x in batch]),
        "attr_labels": {
            col: torch.stack([x["attr_labels"][col] for x in batch])
            for col in ATTRIBUTE_COLS
        },
    }


# ------------------------------------------------------------------------------
# 6. MODEL ARCHITECTURE
# ------------------------------------------------------------------------------

class TaskConditionedKreitlerModel(nn.Module):

    def __init__(self, model_name, num_classes, class_weights):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.bert.resize_token_embeddings(len(tokenizer))

        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)

        self.bio_head = nn.Linear(hidden_size, 3)
        self.ref_start_head = nn.Linear(hidden_size, 1)
        self.ref_end_head = nn.Linear(hidden_size, 1)
        self.attr_heads = nn.ModuleDict({
            col: nn.Linear(hidden_size, num_classes[col])
            for col in ATTRIBUTE_COLS
        })

        self.class_weights = class_weights

    def forward(
        self,
        input_ids,
        attention_mask,
        task_types=None,
        bio_labels=None,
        ref_start=None,
        ref_end=None,
        attr_labels=None,
        **kwargs,
    ):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        cls_output = self.dropout(outputs.pooler_output)

        bio_logits = self.bio_head(sequence_output)
        start_logits = self.ref_start_head(sequence_output).squeeze(-1)
        end_logits = self.ref_end_head(sequence_output).squeeze(-1)
        attr_logits = {
            col: self.attr_heads[col](cls_output) for col in ATTRIBUTE_COLS
        }

        return {
            "bio_logits": bio_logits,
            "start_logits": start_logits,
            "end_logits": end_logits,
            "attr_logits": attr_logits,
        }


# ------------------------------------------------------------------------------
# 7. TRAINER
# ------------------------------------------------------------------------------


class TaskConditionedTrainer(Trainer):

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        task_types = inputs.pop("task_types")
        bio_labels = inputs.pop("bio_labels")
        ref_start = inputs.pop("ref_start")
        ref_end = inputs.pop("ref_end")
        attr_labels = inputs.pop("attr_labels")

        attention_mask = inputs["attention_mask"]
        outputs = model(**inputs)

        device = outputs["bio_logits"].device
        total_loss = torch.tensor(0.0, device=device)

        # Task 1 Loss — Multiplier: 2.0
        m1 = task_types == 1
        if m1.any():
            bio_logits = outputs["bio_logits"][m1]
            bio_target = bio_labels[m1]
            if (bio_target != -100).any():
                bio_fn = nn.CrossEntropyLoss(
                    weight=torch.tensor([1.0, 5.0, 4.0], device=device),
                    ignore_index=-100,
                )
                l1 = bio_fn(bio_logits.view(-1, 3), bio_target.view(-1))
                if torch.isfinite(l1):
                    total_loss = total_loss + (2.0 * l1)

        # Task 2 Loss — Multiplier: 1.5
        m2 = task_types == 2
        if m2.any():
            ce_fn = nn.CrossEntropyLoss(ignore_index=-100)

            s_logits = outputs["start_logits"][m2].clone()
            e_logits = outputs["end_logits"][m2].clone()
            s_target = ref_start[m2]
            e_target = ref_end[m2]
            attn = attention_mask[m2]

            s_logits[attn == 0] = -1e9
            e_logits[attn == 0] = -1e9

            ref_loss = torch.tensor(0.0, device=device)
            has_valid_ref = False

            if (s_target != -100).any():
                l2_s = ce_fn(s_logits, s_target)
                if torch.isfinite(l2_s):
                    ref_loss = ref_loss + l2_s
                    has_valid_ref = True

            if (e_target != -100).any():
                l2_e = ce_fn(e_logits, e_target)
                if torch.isfinite(l2_e):
                    ref_loss = ref_loss + l2_e
                    has_valid_ref = True

            if has_valid_ref:
                total_loss = total_loss + (1.5 * ref_loss)

        # Task 3 Loss — Multiplier: 1.0
        m3 = task_types == 3
        if m3.any():
            attr_total_loss = torch.tensor(0.0, device=device)
            valid_heads = 0
            for col in ATTRIBUTE_COLS:
                logits = outputs["attr_logits"][col][m3]
                labels = attr_labels[col][m3].to(logits.device)

                if (labels != -100).any():
                    weights = model.class_weights[col].to(logits.device)
                    fn = nn.CrossEntropyLoss(
                        weight=weights, ignore_index=-100
                    )
                    h_loss = fn(logits, labels)
                    if torch.isfinite(h_loss):
                        attr_total_loss = attr_total_loss + h_loss
                        valid_heads += 1

            if valid_heads > 0:
                total_loss = total_loss + (attr_total_loss / valid_heads)

        return (total_loss, outputs) if return_outputs else total_loss


# ------------------------------------------------------------------------------
# 8. TRAINING EXECUTION
# ------------------------------------------------------------------------------

train_dataset = TaskConditionedDataset(train_df, tokenizer, encoders, MAX_LEN)
val_dataset = TaskConditionedDataset(val_df, tokenizer, encoders, MAX_LEN)

model = TaskConditionedKreitlerModel(MODEL_NAME, num_classes, class_weights)

batch_size = 16
num_epochs = 6
steps_per_epoch = max(1, len(train_dataset) // batch_size)
total_steps = steps_per_epoch * num_epochs
warmup_steps = int(0.10 * total_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    report_to="none",
)

trainer = TaskConditionedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=task_conditioned_collate_fn,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

print("🚀 Starting Production Task-Conditioned Unified Model Training...")
trainer.train()

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "unified_model.pt"))
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Unified model saved to: {OUTPUT_DIR}")

# ==============================================================================
# PRODUCTION DUAL EVALUATION
# ==============================================================================

import os
import re
import pandas as pd
import torch
from tqdm import tqdm

print("\n" + "=" * 65)
print("📊 EVALUATING TASK-CONDITIONED UNIFIED MODEL")
print("=" * 65)

VAL_FILE = "val_set.csv"
OUTPUT_DIR = "/content/models/task_conditioned_kreitler5"
ATTRIBUTE_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TaskConditionedKreitlerModel(MODEL_NAME, num_classes, class_weights)
model.load_state_dict(
    torch.load(
        os.path.join(OUTPUT_DIR, "unified_model.pt"), map_location=device
    )
)
model.to(device).eval()


def extract_best_span(
    start_logits, end_logits, token_type_ids, max_span_len=12
):
    seq_len = len(start_logits)
    best_score = -float("inf")
    best_s, best_e = 0, 0

    # Restrict span candidates strictly to Context tokens
    for s in range(seq_len):
        if token_type_ids[s] != 1:
            continue
        for e in range(s, min(seq_len, s + max_span_len)):
            if token_type_ids[e] != 1:
                continue
            score = start_logits[s] + end_logits[e]
            if score > best_score:
                best_score = score
                best_s, best_e = s, e
    return best_s, best_e


def run_single_ref_pass_exact(
    sentence, prompt_word, meaning_value, start_char=None, end_char=None
):
    # Exact character slice tagging to handle duplicate MVs cleanly
    if start_char is not None and end_char is not None:
        marked_sent = (
            sentence[:start_char]
            + f"<mv> {sentence[start_char:end_char]} </mv>"
            + sentence[end_char:]
        )
    else:
        marked_sent = (
            sentence.replace(meaning_value, f"<mv> {meaning_value} </mv>", 1)
            if meaning_value in sentence
            else sentence
        )

    context = f"[TASK:REF] גירוי: {prompt_word} | משפט: {marked_sent}"
    question = f"מהו הרפרנט של יחידת המשמעות: {meaning_value} ?"

    enc = tokenizer(
        question,
        context,
        max_length=160,
        truncation="only_second",
        return_offsets_mapping=True,
        return_token_type_ids=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in enc.items() if k != "offset_mapping"}

    with torch.no_grad():
        out = model(**inputs)

    s_log = out["start_logits"][0].cpu().numpy()
    e_log = out["end_logits"][0].cpu().numpy()
    attn = inputs["attention_mask"][0].cpu().numpy()
    type_ids = enc["token_type_ids"][0].cpu().numpy()

    # Mask out padding and Question prompt tokens from logits
    s_log[(attn == 0) | (type_ids == 0)] = -1e9
    e_log[(attn == 0) | (type_ids == 0)] = -1e9

    p_st, p_en = extract_best_span(s_log, e_log, type_ids)
    t2_ids = enc["input_ids"][0].cpu().tolist()
    pred_ref = tokenizer.decode(
        t2_ids[p_st : p_en + 1], skip_special_tokens=True
    ).strip()
    return re.sub(r"</?mv>", "", pred_ref).strip()


def run_single_attr_pass(sentence, meaning_value, referent):
    text = f"[TASK:ATTR] משפט: {sentence} | יחידה: {meaning_value} | רפרנט: {referent}"
    enc = tokenizer(text, return_tensors="pt", max_length=160, truncation=True)
    inputs = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**inputs)

    attr_preds = {}
    for col in ATTRIBUTE_COLS:
        pid = torch.argmax(out["attr_logits"][col], dim=-1).item()
        attr_preds[col] = decoders[col].get(pid, "N/A")
    return attr_preds


def run_prompt_inference(sentence, prompt_word):
    model.eval()

    t1_text = f"[TASK:MV] משפט: {sentence}"
    t1_enc = tokenizer(
        t1_text, return_tensors="pt", max_length=160, return_offsets_mapping=True
    )
    t1_offsets = t1_enc.pop("offset_mapping")[0].tolist()
    t1_inputs = {k: v.to(device) for k, v in t1_enc.items()}

    with torch.no_grad():
        t1_out = model(**t1_inputs)

    bio_preds = torch.argmax(t1_out["bio_logits"], dim=-1)[0].cpu().numpy()

    mv_spans, curr = [], []
    for idx, (p, (st, en)) in enumerate(zip(bio_preds, t1_offsets)):
        if st == 0 and en == 0:
            continue
        if p == 1:
            if curr:
                mv_spans.append(curr)
            curr = [idx]
        elif p == 2:
            if curr:
                curr.append(idx)
            else:
                curr = [idx]
        else:
            if curr:
                mv_spans.append(curr)
                curr = []
    if curr:
        mv_spans.append(curr)

    pred_mvs = []
    prefix_len = len("[TASK:MV] משפט: ")
    for sp in mv_spans:
        st_c = t1_offsets[sp[0]][0] - prefix_len
        en_c = t1_offsets[sp[-1]][1] - prefix_len
        if st_c >= 0 and en_c <= len(sentence):
            txt = sentence[st_c:en_c].strip()
            if txt:
                pred_mvs.append({"text": txt, "start": st_c, "end": en_c})

    results = []
    for mv_obj in pred_mvs:
        mv_text = mv_obj["text"]
        pred_ref = run_single_ref_pass_exact(
            sentence,
            prompt_word,
            mv_text,
            start_char=mv_obj["start"],
            end_char=mv_obj["end"],
        )
        attr_preds = run_single_attr_pass(sentence, mv_text, pred_ref)
        results.append({
            "meaning_value": mv_text,
            "referent": pred_ref,
            "attributes": attr_preds,
        })
    return results


# ------------------------------------------------------------------------------
# EVALUATION EXECUTION: ORACLE VS. END-TO-END CASCADE
# ------------------------------------------------------------------------------

val_df = pd.read_csv(VAL_FILE)
grouped_val = val_df.groupby(["Prompt_Word", "Full_Raw_Sentence"], sort=False)

e2e_t1_correct, e2e_t1_total = 0, 0
e2e_t2_correct, e2e_t2_total = 0, 0
e2e_t3_correct = {col: 0 for col in ATTRIBUTE_COLS}
e2e_t3_total = {col: 0 for col in ATTRIBUTE_COLS}

oracle_t2_correct, oracle_t2_total = 0, 0
oracle_t3_correct = {col: 0 for col in ATTRIBUTE_COLS}
oracle_t3_total = {col: 0 for col in ATTRIBUTE_COLS}

for (prompt_word, sentence), group in tqdm(
    grouped_val, desc="Running Evaluation"
):
    sentence = str(sentence).strip()
    prompt_word = str(prompt_word).strip() if pd.notna(prompt_word) else ""

    gold_mvs = [str(x).strip() for x in group["MeaningVal"].dropna().unique()]
    gold_data = {}
    for _, row in group.iterrows():
        mv = str(row["MeaningVal"]).strip()
        gold_data[mv] = {
            "referent": str(row["Referent"]).strip()
            if pd.notna(row["Referent"])
            else "",
            "attributes": {
                col: str(row[col]).strip() if pd.notna(row[col]) else None
                for col in ATTRIBUTE_COLS
            },
        }

    # 1. ORACLE EVALUATION
    for gold_mv, g_info in gold_data.items():
        o_ref = run_single_ref_pass_exact(sentence, prompt_word, gold_mv)
        if o_ref == g_info["referent"]:
            oracle_t2_correct += 1
        oracle_t2_total += 1

        o_attrs = run_single_attr_pass(sentence, gold_mv, g_info["referent"])
        for col in ATTRIBUTE_COLS:
            if g_info["attributes"][col] is not None:
                oracle_t3_total[col] += 1
                if o_attrs[col] == g_info["attributes"][col]:
                    oracle_t3_correct[col] += 1

# 2. END-TO-END CASCADE EVALUATION
    predictions = run_prompt_inference(sentence, prompt_word)
    pred_mv_texts = [p["meaning_value"] for p in predictions]

    if set(pred_mv_texts) == set(gold_mvs):
        e2e_t1_correct += 1
    e2e_t1_total += 1

    for pred in predictions:
        p_mv = pred["meaning_value"]
        p_ref = pred["referent"]
        p_attrs = pred["attributes"]

        # Track total predicted MVs for Task 2 precision/denominator
        e2e_t2_total += 1

        if p_mv in gold_data:
            g_ref = gold_data[p_mv]["referent"]
            g_attrs = gold_data[p_mv]["attributes"]

            # Task 2 Check
            if p_ref == g_ref:
                e2e_t2_correct += 1

            # Task 3 Check
            for col in ATTRIBUTE_COLS:
                if g_attrs[col] is not None:
                    e2e_t3_total[col] += 1
                    if p_attrs[col] == g_attrs[col]:
                        e2e_t3_correct[col] += 1
        else:
            continue

print("\n" + "=" * 65)
print("🏆 TASK-CONDITIONED UNIFIED MODEL RESULTS")
print("=" * 65)

t1_acc = (e2e_t1_correct / max(e2e_t1_total, 1)) * 100
print(f"Task 1 — Meaning Value Extraction (Exact Set Match): {t1_acc:.2f}%\n")

print("--- TASK 2: REFERENT EXTRACTION ---")
o_t2 = (oracle_t2_correct / max(oracle_t2_total, 1)) * 100
e_t2 = (e2e_t2_correct / max(e2e_t2_total, 1)) * 100
print(f"  • Oracle Accuracy (Gold MV)         : {o_t2:.2f}%")
print(f"  • End-to-End Cascaded Precision    : {e_t2:.2f}%\n")

print("--- TASK 3: PSYCHOLOGICAL CLASSIFICATION ACCURACY ---")
for col in ATTRIBUTE_COLS:
    o_tot = oracle_t3_total[col]
    e_tot = e2e_t3_total[col]
    o_acc = (oracle_t3_correct[col] / o_tot * 100) if o_tot > 0 else 0.0
    e_acc = (e2e_t3_correct[col] / e_tot * 100) if e_tot > 0 else 0.0
    print(
        f"  • {col:<8s} | Oracle: {o_acc:5.2f}% | End-to-End Cascaded:"
        f" {e_acc:5.2f}%"
    )
print("=" * 65)

config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss
1,4.011667,4.153877
2,2.931843,3.569859
3,2.171439,3.744882
4,1.106200,4.370245


✅ Unified model saved to: /content/models/task_conditioned_kreitler5

📊 EVALUATING TASK-CONDITIONED UNIFIED MODEL


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Running Evaluation: 100%|██████████| 808/808 [03:27<00:00,  3.90it/s]


🏆 TASK-CONDITIONED UNIFIED MODEL RESULTS
Task 1 — Meaning Value Extraction (Exact Set Match): 31.93%

--- TASK 2: REFERENT EXTRACTION ---
  • Oracle Accuracy (Gold MV)         : 70.75%
  • End-to-End Cascaded Precision    : 45.58%

--- TASK 3: PSYCHOLOGICAL CLASSIFICATION ACCURACY ---
  • dim      | Oracle: 53.98% | End-to-End Cascaded: 56.45%
  • tr       | Oracle: 77.11% | End-to-End Cascaded: 78.90%
  • fr       | Oracle: 92.94% | End-to-End Cascaded: 93.94%
  • sr       | Oracle: 92.63% | End-to-End Cascaded: 86.54%
  • fe       | Oracle: 99.96% | End-to-End Cascaded: 99.97%
  • refLevel | Oracle: 64.64% | End-to-End Cascaded: 60.48%
  • ss       | Oracle: 64.81% | End-to-End Cascaded: 65.52%


Save model to drive + eval on test

In [ ]:
# ==============================================================================
# TASK-CONDITIONED UNIFIED MODEL
# ==============================================================================

import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from google.colab import drive
from transformers import (
    AutoModel,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# ------------------------------------------------------------------------------
# 1. DRIVE MOUNT & CONFIGURATION
# ------------------------------------------------------------------------------

drive.mount('/content/drive')

MODEL_NAME = "dicta-il/dictabert"
TRAIN_FILE = "train_set.csv" if os.path.exists("train_set.csv") else "stage3_splits/train_set.csv"
VAL_FILE = "val_set.csv" if os.path.exists("val_set.csv") else "stage3_splits/val_set.csv"

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/WORKSHOP/Unified_Model"
LOCAL_TMP_DIR = "/tmp/task_conditioned_checkpoints"

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
MAX_LEN = 160

ATTRIBUTE_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

O = 0
B_MU = 1
I_MU = 2

# ------------------------------------------------------------------------------
# 2. ENCODERS & INVERSE CLASS WEIGHTS
# ------------------------------------------------------------------------------

train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)

encoders = {}
decoders = {}
num_classes = {}

for col in ATTRIBUTE_COLS:
    values = train_df[col].dropna().astype(str).unique().tolist()
    encoders[col] = {value: i for i, value in enumerate(values)}
    decoders[col] = {i: value for value, i in encoders[col].items()}
    num_classes[col] = len(encoders[col])


def compute_class_weights(df, column, encoder):
    valid_values = df[column].dropna().astype(str)
    counts = valid_values.map(encoder).value_counts()
    n_classes = len(encoder)
    total = len(valid_values)

    weights = torch.ones(n_classes, dtype=torch.float)
    for class_id in range(n_classes):
        count = counts.get(class_id, 0)
        if count > 0:
            weights[class_id] = total / (n_classes * count)

    return torch.clamp(weights, min=0.1, max=10.0)


class_weights = {
    col: compute_class_weights(train_df, col, encoders[col])
    for col in ATTRIBUTE_COLS
}

# ------------------------------------------------------------------------------
# 3. TOKENIZER SETUP & SPECIAL PROMPT TOKENS
# ------------------------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
SPECIAL_TOKENS = {
    "additional_special_tokens": [
        "<mv>",
        "</mv>",
        "[TASK:MV]",
        "[TASK:REF]",
        "[TASK:ATTR]",
    ]
}
tokenizer.add_special_tokens(SPECIAL_TOKENS)

# ------------------------------------------------------------------------------
# 4. DATASET WITH REFERENT ALIGNMENT
# ------------------------------------------------------------------------------


class TaskConditionedDataset(Dataset):

    def __init__(self, df, tokenizer, encoders, max_len=160):
        self.tokenizer = tokenizer
        self.encoders = encoders
        self.max_len = max_len
        self.examples = []

        grouped = df.groupby(["Prompt_Word", "Full_Raw_Sentence"], sort=False)

        for (prompt_word, sentence), group in grouped:
            if pd.isna(sentence):
                continue
            sentence = str(sentence).strip()
            prompt_word = (
                str(prompt_word).strip() if pd.notna(prompt_word) else ""
            )

            all_mv_spans = []
            for _, row in group.iterrows():
                if pd.notna(row["MeaningVal"]):
                    mv_str = str(row["MeaningVal"]).strip()
                    pos = sentence.find(mv_str)
                    if pos != -1:
                        all_mv_spans.append((pos, pos + len(mv_str)))

            all_mv_spans = list(dict.fromkeys(all_mv_spans))

            if all_mv_spans:
                self.examples.append({
                    "task_type": "mv",
                    "sentence": sentence,
                    "prompt_word": prompt_word,
                    "all_mv_spans": all_mv_spans,
                })

            for _, row in group.iterrows():
                if pd.isna(row["MeaningVal"]):
                    continue
                mv = str(row["MeaningVal"]).strip()
                ref = (
                    str(row["Referent"]).strip()
                    if pd.notna(row["Referent"])
                    else ""
                )

                self.examples.append({
                    "task_type": "ref",
                    "sentence": sentence,
                    "prompt_word": prompt_word,
                    "meaning_val": mv,
                    "referent": ref,
                })

                self.examples.append({
                    "task_type": "attr",
                    "sentence": sentence,
                    "meaning_val": mv,
                    "referent": ref,
                    "attributes": {col: row[col] for col in ATTRIBUTE_COLS},
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        task_type = ex["task_type"]

        if task_type == "mv":
            text = f"[TASK:MV] משפט: {ex['sentence']}"
            encoding = self.tokenizer(
                text,
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_offsets_mapping=True,
                return_tensors="pt",
            )
            input_ids = encoding["input_ids"].squeeze(0)
            attention_mask = encoding["attention_mask"].squeeze(0)
            offsets = encoding["offset_mapping"].squeeze(0).tolist()

            prefix_len = len("[TASK:MV] משפט: ")
            sentence_st = prefix_len
            sentence_en = prefix_len + len(ex["sentence"])

            bio_labels = torch.full((self.max_len,), -100, dtype=torch.long)
            for t_idx, (st, en) in enumerate(offsets):
                if st == 0 and en == 0:
                    continue
                if en <= sentence_st or st >= sentence_en:
                    continue
                bio_labels[t_idx] = O

            for mv_st, mv_en in ex["all_mv_spans"]:
                mv_st += sentence_st
                mv_en += sentence_st
                first = True
                for t_idx, (st, en) in enumerate(offsets):
                    if st == 0 and en == 0:
                        continue
                    if en > mv_st and st < mv_en:
                        bio_labels[t_idx] = B_MU if first else I_MU
                        first = False

            return {
                "task_type": 1,
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "bio_labels": bio_labels,
                "ref_start": torch.tensor(-100, dtype=torch.long),
                "ref_end": torch.tensor(-100, dtype=torch.long),
                "attr_labels": {
                    col: torch.tensor(-100, dtype=torch.long)
                    for col in ATTRIBUTE_COLS
                },
            }

        elif task_type == "ref":
            mv = ex["meaning_val"]
            sent = ex["sentence"]
            marked_sent = (
                sent.replace(mv, f"<mv> {mv} </mv>", 1) if mv in sent else sent
            )

            context = f"[TASK:REF] גירוי: {ex['prompt_word']} | משפט: {marked_sent}"
            question = f"מהו הרפרנט של יחידת המשמעות: {mv} ?"

            encoding = self.tokenizer(
                question,
                context,
                max_length=self.max_len,
                padding="max_length",
                truncation="only_second",
                return_offsets_mapping=True,
                return_tensors="pt",
            )
            input_ids = encoding["input_ids"].squeeze(0)
            attention_mask = encoding["attention_mask"].squeeze(0)

            ref = ex["referent"]
            ref_st, ref_en = -100, -100

            if ref and ref.strip():
                ref_c_st = context.find(ref)
                if ref_c_st != -1:
                    ref_c_en = ref_c_st + len(ref)
                    st_tok = encoding.char_to_token(
                        0, ref_c_st, sequence_index=1
                    )
                    en_tok = encoding.char_to_token(
                        0, ref_c_en - 1, sequence_index=1
                    )
                    ref_st = st_tok if st_tok is not None else -100
                    ref_en = en_tok if en_tok is not None else -100

            return {
                "task_type": 2,
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "bio_labels": torch.full((self.max_len,), -100, dtype=torch.long),
                "ref_start": torch.tensor(ref_st, dtype=torch.long),
                "ref_end": torch.tensor(ref_en, dtype=torch.long),
                "attr_labels": {
                    col: torch.tensor(-100, dtype=torch.long)
                    for col in ATTRIBUTE_COLS
                },
            }

        else:
            text = f"[TASK:ATTR] משפט: {ex['sentence']} | יחידה: {ex['meaning_val']} | רפרנט: {ex['referent']}"
            encoding = self.tokenizer(
                text,
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            input_ids = encoding["input_ids"].squeeze(0)
            attention_mask = encoding["attention_mask"].squeeze(0)

            attr_labels = {}
            for col in ATTRIBUTE_COLS:
                val = ex["attributes"][col]
                if pd.isna(val):
                    attr_labels[col] = torch.tensor(-100, dtype=torch.long)
                else:
                    cid = self.encoders[col].get(str(val), -100)
                    attr_labels[col] = torch.tensor(cid, dtype=torch.long)

            return {
                "task_type": 3,
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "bio_labels": torch.full((self.max_len,), -100, dtype=torch.long),
                "ref_start": torch.tensor(-100, dtype=torch.long),
                "ref_end": torch.tensor(-100, dtype=torch.long),
                "attr_labels": attr_labels,
            }


def task_conditioned_collate_fn(batch):
    return {
        "task_types": torch.tensor(
            [x["task_type"] for x in batch], dtype=torch.long
        ),
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "bio_labels": torch.stack([x["bio_labels"] for x in batch]),
        "ref_start": torch.stack([x["ref_start"] for x in batch]),
        "ref_end": torch.stack([x["ref_end"] for x in batch]),
        "attr_labels": {
            col: torch.stack([x["attr_labels"][col] for x in batch])
            for col in ATTRIBUTE_COLS
        },
    }


# ------------------------------------------------------------------------------
# 5. MODEL ARCHITECTURE & TRAINER
# ------------------------------------------------------------------------------


class TaskConditionedKreitlerModel(nn.Module):

    def __init__(self, model_name, num_classes, class_weights):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.bert.resize_token_embeddings(len(tokenizer))

        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)

        self.bio_head = nn.Linear(hidden_size, 3)
        self.ref_start_head = nn.Linear(hidden_size, 1)
        self.ref_end_head = nn.Linear(hidden_size, 1)
        self.attr_heads = nn.ModuleDict({
            col: nn.Linear(hidden_size, num_classes[col])
            for col in ATTRIBUTE_COLS
        })

        self.class_weights = class_weights

    def forward(
        self,
        input_ids,
        attention_mask,
        task_types=None,
        bio_labels=None,
        ref_start=None,
        ref_end=None,
        attr_labels=None,
        **kwargs,
    ):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        cls_output = self.dropout(outputs.pooler_output)

        bio_logits = self.bio_head(sequence_output)
        start_logits = self.ref_start_head(sequence_output).squeeze(-1)
        end_logits = self.ref_end_head(sequence_output).squeeze(-1)
        attr_logits = {
            col: self.attr_heads[col](cls_output) for col in ATTRIBUTE_COLS
        }

        return {
            "bio_logits": bio_logits,
            "start_logits": start_logits,
            "end_logits": end_logits,
            "attr_logits": attr_logits,
        }


class TaskConditionedTrainer(Trainer):

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        task_types = inputs.pop("task_types")
        bio_labels = inputs.pop("bio_labels")
        ref_start = inputs.pop("ref_start")
        ref_end = inputs.pop("ref_end")
        attr_labels = inputs.pop("attr_labels")

        attention_mask = inputs["attention_mask"]
        outputs = model(**inputs)

        device = outputs["bio_logits"].device
        total_loss = torch.tensor(0.0, device=device)

        m1 = task_types == 1
        if m1.any():
            bio_logits = outputs["bio_logits"][m1]
            bio_target = bio_labels[m1]
            if (bio_target != -100).any():
                bio_fn = nn.CrossEntropyLoss(
                    weight=torch.tensor([1.0, 5.0, 4.0], device=device),
                    ignore_index=-100,
                )
                l1 = bio_fn(bio_logits.view(-1, 3), bio_target.view(-1))
                if torch.isfinite(l1):
                    total_loss = total_loss + (2.0 * l1)

        m2 = task_types == 2
        if m2.any():
            ce_fn = nn.CrossEntropyLoss(ignore_index=-100)

            s_logits = outputs["start_logits"][m2].clone()
            e_logits = outputs["end_logits"][m2].clone()
            s_target = ref_start[m2]
            e_target = ref_end[m2]
            attn = attention_mask[m2]

            s_logits[attn == 0] = -1e9
            e_logits[attn == 0] = -1e9

            ref_loss = torch.tensor(0.0, device=device)
            has_valid_ref = False

            if (s_target != -100).any():
                l2_s = ce_fn(s_logits, s_target)
                if torch.isfinite(l2_s):
                    ref_loss = ref_loss + l2_s
                    has_valid_ref = True

            if (e_target != -100).any():
                l2_e = ce_fn(e_logits, e_target)
                if torch.isfinite(l2_e):
                    ref_loss = ref_loss + l2_e
                    has_valid_ref = True

            if has_valid_ref:
                total_loss = total_loss + (1.5 * ref_loss)

        m3 = task_types == 3
        if m3.any():
            attr_total_loss = torch.tensor(0.0, device=device)
            valid_heads = 0
            for col in ATTRIBUTE_COLS:
                logits = outputs["attr_logits"][col][m3]
                labels = attr_labels[col][m3].to(logits.device)

                if (labels != -100).any():
                    weights = model.class_weights[col].to(logits.device)
                    fn = nn.CrossEntropyLoss(
                        weight=weights, ignore_index=-100
                    )
                    h_loss = fn(logits, labels)
                    if torch.isfinite(h_loss):
                        attr_total_loss = attr_total_loss + h_loss
                        valid_heads += 1

            if valid_heads > 0:
                total_loss = total_loss + (attr_total_loss / valid_heads)

        return (total_loss, outputs) if return_outputs else total_loss


# ------------------------------------------------------------------------------
# 6. TRAINING EXECUTION
# ------------------------------------------------------------------------------

train_dataset = TaskConditionedDataset(train_df, tokenizer, encoders, MAX_LEN)
val_dataset = TaskConditionedDataset(val_df, tokenizer, encoders, MAX_LEN)

model = TaskConditionedKreitlerModel(MODEL_NAME, num_classes, class_weights)

batch_size = 16
num_epochs = 6
steps_per_epoch = max(1, len(train_dataset) // batch_size)
total_steps = steps_per_epoch * num_epochs
warmup_steps = int(0.10 * total_steps)

training_args = TrainingArguments(
    output_dir=LOCAL_TMP_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    report_to="none",
)

trainer = TaskConditionedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=task_conditioned_collate_fn,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

print("🚀 Starting Task-Conditioned Unified Model Training...")
trainer.train()

print(f"💾 Saving trained model and tokenizer to Drive: {DRIVE_OUTPUT_DIR}...")
torch.save(model.state_dict(), os.path.join(DRIVE_OUTPUT_DIR, "unified_model.pt"))
tokenizer.save_pretrained(DRIVE_OUTPUT_DIR)

import json
with open(os.path.join(DRIVE_OUTPUT_DIR, "decoders.json"), "w", encoding="utf-8") as f:
    json.dump({col: {str(k): v for k, v in decoders[col].items()} for col in decoders}, f, ensure_ascii=False, indent=2)

print(f"✅ Unified model saved to Google Drive: {DRIVE_OUTPUT_DIR}")


# ==============================================================================
# EVALUATION
# ==============================================================================

import os
import re
import json
import pandas as pd
import torch
from tqdm import tqdm

print("\n" + "=" * 65)
print("📊 EVALUATING FINE-TUNED TASK-CONDITIONED MODEL ON TEST SET")
print("=" * 65)

TEST_FILE = "test_set.csv" if os.path.exists("test_set.csv") else "stage3_splits/test_set.csv"
ATTRIBUTE_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TaskConditionedKreitlerModel(MODEL_NAME, num_classes, class_weights)
model.load_state_dict(
    torch.load(
        os.path.join(DRIVE_OUTPUT_DIR, "unified_model.pt"), map_location=device
    )
)
model.to(device).eval()


def extract_best_span(
    start_logits, end_logits, token_type_ids, max_span_len=12
):
    seq_len = len(start_logits)
    best_score = -float("inf")
    best_s, best_e = 0, 0

    for s in range(seq_len):
        if token_type_ids[s] != 1:
            continue
        for e in range(s, min(seq_len, s + max_span_len)):
            if token_type_ids[e] != 1:
                continue
            score = start_logits[s] + end_logits[e]
            if score > best_score:
                best_score = score
                best_s, best_e = s, e
    return best_s, best_e


def run_single_ref_pass_exact(
    sentence, prompt_word, meaning_value, start_char=None, end_char=None
):
    if start_char is not None and end_char is not None:
        marked_sent = (
            sentence[:start_char]
            + f"<mv> {sentence[start_char:end_char]} </mv>"
            + sentence[end_char:]
        )
    else:
        marked_sent = (
            sentence.replace(meaning_value, f"<mv> {meaning_value} </mv>", 1)
            if meaning_value in sentence
            else sentence
        )

    context = f"[TASK:REF] גירוי: {prompt_word} | משפט: {marked_sent}"
    question = f"מהו הרפרנט של יחידת המשמעות: {meaning_value} ?"

    enc = tokenizer(
        question,
        context,
        max_length=160,
        truncation="only_second",
        return_offsets_mapping=True,
        return_token_type_ids=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in enc.items() if k != "offset_mapping"}

    with torch.no_grad():
        out = model(**inputs)

    s_log = out["start_logits"][0].cpu().numpy()
    e_log = out["end_logits"][0].cpu().numpy()
    attn = inputs["attention_mask"][0].cpu().numpy()
    type_ids = enc["token_type_ids"][0].cpu().numpy()

    s_log[(attn == 0) | (type_ids == 0)] = -1e9
    e_log[(attn == 0) | (type_ids == 0)] = -1e9

    p_st, p_en = extract_best_span(s_log, e_log, type_ids)
    t2_ids = enc["input_ids"][0].cpu().tolist()
    pred_ref = tokenizer.decode(
        t2_ids[p_st : p_en + 1], skip_special_tokens=True
    ).strip()
    return re.sub(r"</?mv>", "", pred_ref).strip()


def run_single_attr_pass(sentence, meaning_value, referent):
    text = f"[TASK:ATTR] משפט: {sentence} | יחידה: {meaning_value} | רפרנט: {referent}"
    enc = tokenizer(text, return_tensors="pt", max_length=160, truncation=True)
    inputs = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**inputs)

    attr_preds = {}
    for col in ATTRIBUTE_COLS:
        pid = torch.argmax(out["attr_logits"][col], dim=-1).item()
        attr_preds[col] = decoders[col].get(pid, "N/A")
    return attr_preds


def run_prompt_inference(sentence, prompt_word):
    model.eval()

    t1_text = f"[TASK:MV] משפט: {sentence}"
    t1_enc = tokenizer(
        t1_text, return_tensors="pt", max_length=160, return_offsets_mapping=True
    )
    t1_offsets = t1_enc.pop("offset_mapping")[0].tolist()
    t1_inputs = {k: v.to(device) for k, v in t1_enc.items()}

    with torch.no_grad():
        t1_out = model(**t1_inputs)

    bio_preds = torch.argmax(t1_out["bio_logits"], dim=-1)[0].cpu().numpy()

    mv_spans, curr = [], []
    for idx, (p, (st, en)) in enumerate(zip(bio_preds, t1_offsets)):
        if st == 0 and en == 0:
            continue
        if p == 1:
            if curr:
                mv_spans.append(curr)
            curr = [idx]
        elif p == 2:
            if curr:
                curr.append(idx)
            else:
                curr = [idx]
        else:
            if curr:
                mv_spans.append(curr)
                curr = []
    if curr:
        mv_spans.append(curr)

    pred_mvs = []
    prefix_len = len("[TASK:MV] משפט: ")
    for sp in mv_spans:
        st_c = t1_offsets[sp[0]][0] - prefix_len
        en_c = t1_offsets[sp[-1]][1] - prefix_len
        if st_c >= 0 and en_c <= len(sentence):
            txt = sentence[st_c:en_c].strip()
            if txt:
                pred_mvs.append({"text": txt, "start": st_c, "end": en_c})

    results = []
    for mv_obj in pred_mvs:
        mv_text = mv_obj["text"]
        pred_ref = run_single_ref_pass_exact(
            sentence,
            prompt_word,
            mv_text,
            start_char=mv_obj["start"],
            end_char=mv_obj["end"],
        )
        attr_preds = run_single_attr_pass(sentence, mv_text, pred_ref)
        results.append({
            "meaning_value": mv_text,
            "referent": pred_ref,
            "attributes": attr_preds,
        })
    return results


# ------------------------------------------------------------------------------
# EVALUATION EXECUTION
# ------------------------------------------------------------------------------

test_df = pd.read_csv(TEST_FILE)
grouped_test = test_df.groupby(["Prompt_Word", "Full_Raw_Sentence"], sort=False)

e2e_t1_correct, e2e_t1_total = 0, 0
e2e_t2_correct, e2e_t2_total = 0, 0
e2e_t3_correct = {col: 0 for col in ATTRIBUTE_COLS}
e2e_t3_total = {col: 0 for col in ATTRIBUTE_COLS}

oracle_t2_correct, oracle_t2_total = 0, 0
oracle_t3_correct = {col: 0 for col in ATTRIBUTE_COLS}
oracle_t3_total = {col: 0 for col in ATTRIBUTE_COLS}

for (prompt_word, sentence), group in tqdm(
    grouped_test, desc="Running Test Evaluation"
):
    sentence = str(sentence).strip()
    prompt_word = str(prompt_word).strip() if pd.notna(prompt_word) else ""

    gold_mvs = [str(x).strip() for x in group["MeaningVal"].dropna().unique()]
    gold_data = {}
    for _, row in group.iterrows():
        mv = str(row["MeaningVal"]).strip()
        gold_data[mv] = {
            "referent": str(row["Referent"]).strip()
            if pd.notna(row["Referent"])
            else "",
            "attributes": {
                col: str(row[col]).strip() if pd.notna(row[col]) else None
                for col in ATTRIBUTE_COLS
            },
        }

    # 1. ORACLE EVALUATION
    for gold_mv, g_info in gold_data.items():
        o_ref = run_single_ref_pass_exact(sentence, prompt_word, gold_mv)
        if o_ref == g_info["referent"]:
            oracle_t2_correct += 1
        oracle_t2_total += 1

        o_attrs = run_single_attr_pass(sentence, gold_mv, g_info["referent"])
        for col in ATTRIBUTE_COLS:
            if g_info["attributes"][col] is not None:
                oracle_t3_total[col] += 1
                if o_attrs[col] == g_info["attributes"][col]:
                    oracle_t3_correct[col] += 1

    # 2. END-TO-END CASCADE EVALUATION
    predictions = run_prompt_inference(sentence, prompt_word)
    pred_mv_texts = [p["meaning_value"] for p in predictions]

    if set(pred_mv_texts) == set(gold_mvs):
        e2e_t1_correct += 1
    e2e_t1_total += 1

    for pred in predictions:
        p_mv = pred["meaning_value"]
        p_ref = pred["referent"]
        p_attrs = pred["attributes"]

        e2e_t2_total += 1

        if p_mv in gold_data:
            g_ref = gold_data[p_mv]["referent"]
            g_attrs = gold_data[p_mv]["attributes"]

            if p_ref == g_ref:
                e2e_t2_correct += 1

            for col in ATTRIBUTE_COLS:
                if g_attrs[col] is not None:
                    e2e_t3_total[col] += 1
                    if p_attrs[col] == g_attrs[col]:
                        e2e_t3_correct[col] += 1
        else:
            continue

print("\n" + "=" * 65)
print("🏆 TASK-CONDITIONED UNIFIED MODEL TEST SET RESULTS")
print("=" * 65)

t1_acc = (e2e_t1_correct / max(e2e_t1_total, 1)) * 100
print(f"Task 1 — Meaning Value Extraction (Exact Set Match): {t1_acc:.2f}%\n")

print("--- TASK 2: REFERENT EXTRACTION ---")
o_t2 = (oracle_t2_correct / max(oracle_t2_total, 1)) * 100
e_t2 = (e2e_t2_correct / max(e2e_t2_total, 1)) * 100
print(f"  • Oracle Accuracy (Gold MV)         : {o_t2:.2f}%")
print(f"  • End-to-End Cascaded Precision    : {e_t2:.2f}%\n")

print("--- TASK 3: PSYCHOLOGICAL CLASSIFICATION ACCURACY ---")
for col in ATTRIBUTE_COLS:
    o_tot = oracle_t3_total[col]
    e_tot = e2e_t3_total[col]
    o_acc = (oracle_t3_correct[col] / o_tot * 100) if o_tot > 0 else 0.0
    e_acc = (e2e_t3_correct[col] / e_tot * 100) if e_tot > 0 else 0.0
    print(
        f"  • {col:<8s} | Oracle: {o_acc:5.2f}% | End-to-End Cascaded:"
        f" {e_acc:5.2f}%"
    )
print("=" * 65)

Mounted at /content/drive


config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

...

Epoch,Training Loss,Validation Loss
1,4.014154,4.146827
2,3.062481,3.607283
3,1.999092,3.881455
4,1.111092,4.665534


💾 Saving trained model and tokenizer to Drive: /content/drive/MyDrive/WORKSHOP/Unified_Model...
✅ Unified model saved to Google Drive: /content/drive/MyDrive/WORKSHOP/Unified_Model

📊 EVALUATING FINE-TUNED TASK-CONDITIONED MODEL ON TEST SET


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Running Test Evaluation: 100%|██████████| 800/800 [03:16<00:00,  4.07it/s]


🏆 TASK-CONDITIONED UNIFIED MODEL TEST SET RESULTS
Task 1 — Meaning Value Extraction (Exact Set Match): 34.12%

--- TASK 2: REFERENT EXTRACTION ---
  • Oracle Accuracy (Gold MV)         : 71.55%
  • End-to-End Cascaded Precision    : 48.46%

--- TASK 3: PSYCHOLOGICAL CLASSIFICATION ACCURACY ---
  • dim      | Oracle: 52.40% | End-to-End Cascaded: 54.15%
  • tr       | Oracle: 76.62% | End-to-End Cascaded: 79.16%
  • fr       | Oracle: 94.03% | End-to-End Cascaded: 95.08%
  • sr       | Oracle: 93.62% | End-to-End Cascaded: 86.84%
  • fe       | Oracle: 100.00% | End-to-End Cascaded: 100.00%
  • refLevel | Oracle: 67.51% | End-to-End Cascaded: 62.84%
  • ss       | Oracle: 66.67% | End-to-End Cascaded: 67.21%
